# The workbench

The live devising path, one call at a time, with everything it sends and everything it gets back on the page. Nothing here is a copy: the prompts are read out of `agents/` and `shared/`, and the calls are the ones `panel/devising.py` and `panel/continuing.py` make.

Where the product loops on its own — a repair, a check, the gate — there is a cell, so what it does silently in a container is visible here in order.

One afternoon is n=1. What to try is decided here; whether it worked is answered by `python -m research.run`.

In [1]:
import os
import secrets
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import research.bench as bench  # noqa: E402
from agents.experience_deviser import ExperienceDeviser, experience_in, the_prompt  # noqa: E402
from orchestrator.router import FoundryConfig, FoundryRouter  # noqa: E402
from orchestrator.safety import (  # noqa: E402
    AzureContentSafetyGate,
    ContentSafetyConfig,
    screen_experience,
)
from shared.agents import AgentContext  # noqa: E402
from shared.ids import LearnerId  # noqa: E402
from shared.seal import Sealer, SealPurpose  # noqa: E402

bench.environment()

# Built exactly as `panel/devising.py` builds it, so the gate and the router here are the
# ones production uses. The safety key is per-process: the seal this gate mints travels
# nowhere, on this path or on that one.
ENVIRONMENT = dict(os.environ)
KEY = ENVIRONMENT.get("LANTERNINA_SAFETY_KEY", "").encode() or secrets.token_bytes(32)
GATE = AzureContentSafetyGate(
    ContentSafetyConfig.from_env(ENVIRONMENT),
    Sealer(SealPurpose.CONTENT_SAFETY, KEY, "orchestrator.safety"),
)
ROUTER = FoundryRouter(FoundryConfig.from_env(ENVIRONMENT), gate=GATE)
CTX = AgentContext(router=ROUTER, learner_id=LearnerId(""), learner_hints={}, now=time.time())
DEVISER = ExperienceDeviser()

print("prompt fingerprint:", bench.reload_prompts())

prompt fingerprint: bdbfb5b564f4


## The calls, in order

Six, and every one of them is the call the container makes. `panel/devising.py` runs 1 to 4 inside one request and returns; the house walks the moments and 5 happens at every `hand_over`; `panel/continuing.py` runs 6 when a page comes back off the glass and the outcome says `ask`.

| # | call | to | what it is sent | what comes back | measured |
| --- | --- | --- | --- | --- | --- |
| 1 | which method | the model that writes | `choosing`, and a sample of 60 method names — no records | two ids and a reason | a few seconds |
| 2 | the afternoon | the model that writes | the whole standing instruction plus the household's material, about 34 000 characters | one JSON document: title, overview, themes, script, minutes, drawn, moments | 76–184 s, median about 140 |
| 3a | `repair_unreadable`, only if the format will not read it | the model that writes | `repair`, the refusal, and the answer as it came | the answer again | |
| 3b | `repair`, only if a check refuses it — once and no more | the model that writes | `repair`, the complaints, and the document | the document again | |
| 4 | the gate | the safety gate | the document's own words | through, or `SafetyBlocked` | seconds |
| 5 | the page, at every `hand_over` | the model that draws | `page_maker.*`, built from the page's own words | one PNG | 25–35 s a page |
| 6 | the continuation, when an outcome says `ask` | the model that writes | `experience_continuer.*`, the afternoon so far and what came back on the sheet | the rest of the moments | 15–25 s |

A role and not a deployment: which model answers each one is the router's business, and `research/env.ps1` says which is deployed today.

The stand-in is the one call in this notebook the house never makes: it is `research/calls.adolescent.md`, and it exists so an afternoon can be walked with nobody in the room.

**Four things this notebook does differently, and none of them changes what is sent.** A failed choosing call raises here and is drawn around in the container. A document the checks still refuse after its one repair raises `RefusedByTheChecks` in the container and is only printed here. The gate is not closed at the end. And the walk — the stand-in, and stepping through the moments — is `research/`, not the product: in a house a person does that.

Credentials are not in `env.ps1`: the router builds a `DefaultAzureCredential`, which finds the `az` session in the `AZURE_CONFIG_DIR` named there. If a call fails on authentication, run `az account get-access-token` in a terminal with that variable set — `az account show` reads the cache and succeeds even on an expired token.

In [2]:
from IPython.display import Markdown, display

# Mermaid renders in a cell's output and not in a markdown cell: the renderer that ships
# with VS Code reads `env.outputItem`, which a markdown cell has not got.
display(
    Markdown("""
```mermaid
sequenceDiagram
  autonumber
  participant P as panel/devising
  participant M as the model that writes
  participant S as the safety gate
  P->>M: which method — 60 names out of methods/
  M-->>P: a form and a move
  P->>M: the afternoon — about 34 000 characters
  M-->>P: one JSON document
  opt the format will not read it
    P->>M: repair_unreadable — the refusal and the answer
    M-->>P: the answer again
  end
  opt a check refuses it, once and no more
    P->>M: repair — the complaints and the document
    M-->>P: the document again
  end
  P->>S: the document's own words
  S-->>P: through, or SafetyBlocked
```
""")
)


```mermaid
sequenceDiagram
  autonumber
  participant P as panel/devising
  participant M as the model that writes
  participant S as the safety gate
  P->>M: which method — 60 names out of methods/
  M-->>P: a form and a move
  P->>M: the afternoon — about 34 000 characters
  M-->>P: one JSON document
  opt the format will not read it
    P->>M: repair_unreadable — the refusal and the answer
    M-->>P: the answer again
  end
  opt a check refuses it, once and no more
    P->>M: repair — the complaints and the document
    M-->>P: the document again
  end
  P->>S: the document's own words
  S-->>P: through, or SafetyBlocked
```


## 1. The prompts, as they are

Every block, with the format's own numbers filled in. The placeholders left as `$name` are the ones that carry a household's material, and the cell after next is where they are filled.

In [3]:
P = bench.everything()

for key, text in P.items():
    print(f"{key:26} {len(text):6}")
print(f"{'':26} {sum(len(one) for one in P.values()):6}  in all")

task                         2077
format                       4935
shape-of-a-moment            2765
acts                         1099
marks-on-a-page              3466
ten-dimensions               1135
rules-head                     88
limits                        482
rules-tail                    713
asking                        376
manner-head                   391
how-the-text-reads            860
what-to-refuse                421
only-what-you-can-answer     1399
worth-doing                  5117
manner-tail                   114
method                        810
household                    1391
pitch                         516
stand-in                     1535
                            29690  in all


In [4]:
print(P["task"])

You are making one afternoon for one adolescent to spend at home, mostly on their own, in a house with a printer, a scanner and a small screen or two. Not a lesson, not a test, and not an exercise with a story painted over it. One thing worth doing, that happens once, and is over when it is over.

A parent glances at your overview before this happens, and may add something of their own. That is not a tribunal and you are not defending anything: they are seeing roughly what is coming, for somebody they know and you do not. Write the overview so that one glance says what this afternoon actually is. Nothing here is set in stone either — an afternoon can turn out differently once it has started, and that is ordinary rather than a failure.

An afternoon is a good one when four things are true of it.

Somebody can start it alone. The first screen puts a situation in front of them, they can see what to pick up, and no adult has to explain anything first.

Something is genuinely not known, and

## 2. The household

Everything that decides one afternoon, as one dictionary. `research/run.py` sends the same one to `devise_experience`.

In [5]:
from research.households import Household, Memory, arguments

HOUSE = Household(
    name="bench",
    interests=("i treni", "le mappe vecchie"),
    avoid=("i ragni",),
    load="middle",
    ink="middle",
    span="middle",
    sheets=2,
    note="",
)
MEMORY = Memory()  # no history: the first afternoon of this house
ARGS = arguments(HOUSE, MEMORY)

for name, value in ARGS.items():
    shown = value if isinstance(value, str) else repr(value)
    print(f"{name:12} {shown[:110] or '—'}")

capabilities frozenset({<HouseCapability.PRINT_A4: 'print_a4'>, <HouseCapability.SHOW_800X480_1BIT: 'show_800x480_1bit'>, <
language     Italian
interests    ('i treni', 'le mappe vecchie')
avoid        ('i ragni',)
pitch        two things that have to be put side by side before either makes sense, and one turn where what looked true sto
sheets       2
note         —
already      ()
recent       ()
happened     —
counts       {"afternoonsRun": 0, "ranToTheEnd": 0, "endedEarly": 0, "stopped": 0, "sheetsWrittenOn": 0, "sheetsBlank": 0}
direction    Ask for about what the last ones asked for. Nothing here says to move either way.
ground       —


## 3. Call one: which method

A cheap call before the expensive one. It carries a sample of sixty names out of `methods/` and no records, so the corpus stays outside the prompt. The sample is there because the whole catalogue made the model pick the same pair twice out of twice: the same judgement on the same list gives the same answer, which was worse than drawing.

In [6]:
from shared.methods import CATALOGUE, by_id, draw, index, load, runnable

RUNNABLE = runnable(load(), capabilities=ARGS["capabilities"])
CATALOGUE_SENT = index(RUNNABLE, sample=CATALOGUE)
print(f"{len(RUNNABLE)} methods this house can run · {len(CATALOGUE_SENT)} characters of names\n")
print(CATALOGUE_SENT[:1200])

145 methods this house can run · 5119 characters of names

solids-given-as-layers-of-squares: read a solid off three views and count what fits
a-budget-and-what-each-unit-earned: spend a fixed budget, then write what each unit earned
pieces-that-come-out-exact: cut the pieces and make them come out exact
one-guess-you-expect-to-be-refused: make them offer one case they expect to be refused (a move)
say-nobody-checks-and-print-where-it-goes: say nobody will check it, and print where it goes (a move)
rows-a-house-can-answer: rows that name a property and let the house supply the thing
fourteen-rows-that-lead-nowhere: verify with twenty printed rows, fourteen of them decoys (a move)
edge-that-continues-the-background: trace the edge that continues the background
the-drawing-that-straightens-from-one-place: put the eye where the drawing straightens out
a-manual-for-a-thing-that-is-never-named: instructions for a thing that is never named
blank-at-the-end-of-the-line: leave the blank at the

In [7]:
began = time.time()
WANTS_FORM, WANTS_MOVE, WHY = await DEVISER.choose(
    CTX,
    catalogue=CATALOGUE_SENT,
    interests=ARGS["interests"],
    avoid=ARGS["avoid"],
    already=ARGS["already"],
    pitch=ARGS["pitch"],
)
ASKED_FOR = by_id(RUNNABLE, [WANTS_FORM, WANTS_MOVE])
FORM = next((one for one in ASKED_FOR if not one.is_a_move), None)
MOVE = next((one for one in ASKED_FOR if one.is_a_move), None)
if FORM is None or MOVE is None:
    # What `panel/devising.py` does: a step that exists to improve an afternoon may never
    # be the step that costs one.
    FORM, MOVE = draw(RUNNABLE)
    print("the choice did not resolve; drew instead")
print(f"{time.time() - began:.1f} s · {FORM.method_id} + {MOVE.method_id}\n{WHY}")

10.3 s · a-machine-made-of-two-printed-tables + say-the-sheet-itself-can-be-wrong
An old railway map and timetable work only together, until one planted error overturns the route they seemed to prove.


## 4. The whole prompt, before paying for it

In [8]:
PROMPT = the_prompt(**ARGS, form=FORM, move=MOVE)
print(f"{len(PROMPT)} characters\n")
print(PROMPT)

34023 characters

You are making one afternoon for one adolescent to spend at home, mostly on their own, in a house with a printer, a scanner and a small screen or two. Not a lesson, not a test, and not an exercise with a story painted over it. One thing worth doing, that happens once, and is over when it is over.

A parent glances at your overview before this happens, and may add something of their own. That is not a tribunal and you are not defending anything: they are seeing roughly what is coming, for somebody they know and you do not. Write the overview so that one glance says what this afternoon actually is. Nothing here is set in stone either — an afternoon can turn out differently once it has started, and that is ordinary rather than a failure.

An afternoon is a good one when four things are true of it.

Somebody can start it alone. The first screen puts a situation in front of them, they can see what to pick up, and no adult has to explain anything first.

Something is genuin

## 5. Call two: the afternoon

One answer, whole. Measured at 76–184 s, median about 140.

In [9]:
began = time.time()
ANSWER = await DEVISER.ask(CTX, **ARGS, form=FORM, move=MOVE)
print(f"{time.time() - began:.1f} s · {len(ANSWER)} characters back")
print(ROUTER.last_usage)

132.0 s · 12035 characters back
ModelUsage(deployment='gpt-5.6-sol-2026-07-09', request_id='7f231c6b-cc87-49de-b2ec-1e65dc0701a0', input_tokens=7823, output_tokens=9477, cached_input_tokens=0, reasoning_tokens=6144, size='', quality='')


What came back, exactly as it came.

In [10]:
print(ANSWER)

{"title":"La fermata che Nora volle vedere","overview":"Accosta due carte ferroviarie, prolunga i tracciati, confronta le fermate e annota perché Nora ne spostò una. Dalla stampante escono una vecchia mappa e il foglio piegato che la completa. Alla fine resta una carta doppia, segnata e nominata, da conservare. Non occorre trovare altro. Parla di solitudine in una casa di segnalazione.","themes":["ferrovie","mappe antiche","scelte personali","linee di vista"],"script":"THE WORLD\nValnera esiste dove una ferrovia attraversa due volte la stessa ansa. Nella casa di segnalazione la luce entra da una sola finestra e sa di carta umida. Nessuno nomina mai la fermata cancellata, benché compaia su ogni vecchia mappa.\n\nTHE WAY IN\nDal vassoio della stampante affiora una mappa che sembra finire troppo presto. Il bordo porta due tracciati interrotti e una piega lasciata da un altro foglio. Sul tavolo ci sono un foglio bianco e una matita: la prima schermata dice di prendere la matita e tracciare

## 6. The format reads it

In [11]:
from shared.experience import ExperienceError

REFUSAL = ""
try:
    EXPERIENCE = experience_in(ANSWER)
except ExperienceError as exc:
    EXPERIENCE, REFUSAL = None, str(exc)
    print("the format would not read it:", REFUSAL)
else:
    print(f"{EXPERIENCE.title} · {len(EXPERIENCE.moments)} moments · {EXPERIENCE.minutes} min")

the format would not read it: a way out must be a whole number


If it would not read it at all, the answer goes back up with the refusal. `repair_unreadable`.

In [12]:
if REFUSAL:
    began = time.time()
    EXPERIENCE = await DEVISER.repair_unreadable(
        CTX, answer=ANSWER, refusal=REFUSAL, language=ARGS["language"]
    )
    print(f"{time.time() - began:.1f} s · {EXPERIENCE.title}")
else:
    print("the format read it; nothing to repair")

21.0 s · La fermata che Nora volle vedere


## 7. The seven checks

`shared/experience_checks.py`. They read the shape of the document and not its sense, and a complaint that comes back every afternoon names the rule in the prompt to work on.

In [13]:
from shared.experience_checks import check

COMPLAINTS = check(EXPERIENCE, recent=ARGS["recent"], sheets_at_most=ARGS["sheets"])
print("; ".join(map(str, COMPLAINTS)) or "no complaints")

no complaints


One repair and no more, which is what `panel/devising.py` allows.

In [14]:
if COMPLAINTS:
    began = time.time()
    EXPERIENCE = await DEVISER.repair(
        CTX, refused=EXPERIENCE, complaints=COMPLAINTS, language=ARGS["language"]
    )
    COMPLAINTS = check(EXPERIENCE, recent=ARGS["recent"], sheets_at_most=ARGS["sheets"])
    print(f"{time.time() - began:.1f} s · after the repair: ")
    print("; ".join(map(str, COMPLAINTS)) or "no complaints")
else:
    print("nothing to repair")

nothing to repair


## 8. The gate

The chokepoint. Nothing on this path returns past it: the parse and the checks can refuse, but only screening lets something through.

In [15]:
began = time.time()
await screen_experience(GATE, EXPERIENCE, context="devising an afternoon")
DOCUMENT = EXPERIENCE.to_dict()
print(f"{time.time() - began:.1f} s · the gate let it through")

1.5 s · the gate let it through


## 9. What came out

Only what would reach somebody in the room.

In [16]:
from tools.as_it_arrives import read

COUNTED = read(DOCUMENT)

──────────────────────────────────────────────────────────────────────────────
  La fermata che Nora volle vedere
  Accosta due carte ferroviarie, prolunga i tracciati, confronta le fermate e annota perché Nora ne spostò una. Dalla stampante escono una vecchia mappa e il foglio piegato che la completa. Alla fine resta una carta doppia, segnata e nominata, da conservare. Non occorre trovare altro. Parla di solitudine in una casa di segnalazione.
  80 minuti
──────────────────────────────────────────────────────────────────────────────

1. [say] Una carta sta affiorando
    La carta non finisce al margine.
    Una seconda piega le manca.
    Prendi la matita sul tavolo.
    Disegna una curva sul foglio bianco.
      ↳ dopo 2 min: La carta segue una vecchia piega.
      ↳ dopo 5 min: Il foglio bianco aspetta un segno.
      ↳ dopo 9 min: Traccia una curva in qualunque punto.
      ↳ dopo 14 min: Una sola curva apre l’archivio.
      ⇥ via d'uscita, con «la matita»: Una linea basta

2. [ha

And the script, which is what the parent approves and what whoever runs the afternoon reads.

In [17]:
print(EXPERIENCE.script)

THE WORLD Valnera esiste dove una ferrovia attraversa due volte la stessa ansa. Nella casa di segnalazione la luce entra da una sola finestra e sa di carta umida. Nessuno nomina mai la fermata cancellata, benché compaia su ogni vecchia mappa. THE WAY IN Dal vassoio della stampante affiora una mappa che sembra finire troppo presto. Il bordo porta due tracciati interrotti e una piega lasciata da un altro foglio. Sul tavolo ci sono un foglio bianco e una matita: la prima schermata dice di prendere la matita e tracciare una curva sul foglio. THE QUESTION Risposta: Nora spostò la fermata perché voleva vedere i treni dalla finestra della casa di segnalazione. La stabiliscono: l’ansa ripetuta sulle due carte, il percorso continuo che resta più corto, la linea dalla finestra alla fermata sul percorso interrotto. Operazione: accostare, prolungare, confrontare, escludere. Falsa risposta: Nora voleva accorciare la linea; cade quando i bordi accostati mostrano che il tracciato nuovo compie l’ansa 

## 10. The walk

Run the four cells below in order, then run them again, until it is over. Each pass is one moment.

⚠️ The image deployment has capacity 2 in this region, so two page calls within about 30 s answer 429. A page is 25–35 s and a few cents. Which one is deployed today is in `research/env.ps1`, and the cell above prints what the last call cost.

In [18]:
display(
    Markdown("""
```mermaid
sequenceDiagram
  autonumber
  participant H as the house
  participant G as the model that draws
  participant M as the model that writes
  loop one moment at a time
    opt the moment hands a page over
      H->>G: the page, from its own words
      G-->>H: a PNG, printed and left on the table
    end
    opt it collects one and the outcome says ask
      H->>M: the afternoon so far, and what came back on the sheet
      M-->>H: the rest of the moments
    end
  end
```
""")
)


```mermaid
sequenceDiagram
  autonumber
  participant H as the house
  participant G as the model that draws
  participant M as the model that writes
  loop one moment at a time
    opt the moment hands a page over
      H->>G: the page, from its own words
      G-->>H: a PNG, printed and left on the table
    end
    opt it collects one and the outcome says ask
      H->>M: the afternoon so far, and what came back on the sheet
      M-->>H: the rest of the moments
    end
  end
```


In [19]:
from IPython.display import Image, display

from agents.page_maker import PageMaker
from shared.experience import ASK, Collect, HandOver, Say, Weight

WEIGHT = Weight.STANDARD
MOOD = "una giornata normale, c'è voglia di fare qualcosa"

BY_ID = {one.id: one for one in EXPERIENCE.moments}
ORDER = [one.id for one in EXPERIENCE.moments]
AT = ORDER[0]
MINUTES = 0
DISPLAYS: list[str] = []
SHEET = ""
CAME = None


def show(moment) -> None:
    """What this moment puts in the room, and the ladder somebody stuck would meet."""
    global MINUTES, SHEET
    weighing = moment.at(WEIGHT)
    MINUTES += weighing.minutes
    print(f"[{moment.act}] {moment.heading}   ({weighing.minutes} min, {MINUTES} in)")
    for line in weighing.lines:
        print(f"   display: {line}")
        DISPLAYS.append(line)
    for rung in moment.help:
        print(f"     after {rung.after_minutes} min: {' '.join(rung.lines)}")
    if isinstance(moment, HandOver):
        page = moment.page
        SHEET = "\n".join(
            [f"[{page.kind}] {page.title}", *page.note]
            + [f"- {one.label} ({one.room})" for one in page.spaces]
        )
        print(f"\n   page [{page.kind}] {page.title}")
        for line in page.note:
            print(f"     {line}")
        for space in page.spaces:
            print(f"     [ {space.label} — {space.room} ]")
        print(f"     drawing: {page.illustration}")
    if isinstance(moment, Collect):
        print(f"   way out: {moment.way_out.heading} ({moment.way_out.in_hand})")
        for one in moment.outcomes:
            print(f"   if {one.when} -> {one.then}")


def next_id(moment, came: str | None = None) -> str | None:
    """Where the afternoon goes after this. None is an ending, ASK is a continuation."""
    if isinstance(moment, (Say, HandOver)):
        i = ORDER.index(moment.id)
        return ORDER[i + 1] if i + 1 < len(ORDER) else None
    if isinstance(moment, Collect):
        return next((one.then for one in moment.outcomes if str(one.when) == came), ASK)
    return None

**a.** Where we are.

In [20]:
M = BY_ID[AT]
show(M)

[say] Una carta sta affiorando   (7 min, 7 in)
   display: La carta non finisce al margine.
   display: Una seconda piega le manca.
   display: Prendi la matita sul tavolo.
   display: Disegna una curva sul foglio bianco.
     after 2 min: La carta segue una vecchia piega.
     after 5 min: Il foglio bianco aspetta un segno.
     after 9 min: Traccia una curva in qualunque punto.
     after 14 min: Una sola curva apre l’archivio.


**b.** The page, if this moment hands one over.

In [21]:
if isinstance(M, HandOver):
    began = time.time()
    PNG, ASKED = await PageMaker().draw(CTX, M.page)
    (ROOT / "tmp").mkdir(exist_ok=True)
    (ROOT / "tmp" / f"{M.id}.png").write_bytes(PNG)
    print(f"{time.time() - began:.1f} s · {len(PNG)} bytes")
    display(Image(PNG, width=520))
    print(ASKED)
else:
    print(f"{M.act} — nothing to draw here")

say — nothing to draw here


**c.** What the person did with it, if this moment collects one.

In [22]:
from string import Template

from research.calls import what_they_did

if isinstance(M, Collect):
    print(Template(P["stand-in"]).substitute(
        displays="\n".join(DISPLAYS), sheet=SHEET, mood=MOOD, minutes=MINUTES
    ))
    DID = await what_they_did(
        CTX, displays=DISPLAYS, sheet=SHEET, mood=MOOD, minutes_in=MINUTES
    )
    CAME = "marks" if str(DID.get("came")) == "marks" else "blank"
    print(f"\n{CAME} · {DID.get('onIt')} · stop: {DID.get('stop')}\n{DID.get('why')}")
else:
    CAME = None
    print(f"{M.act} — nobody is being asked for anything here")

say — nobody is being asked for anything here


**d.** Take the branch, and go back to **a**.

In [23]:
AT = next_id(M, CAME)
if AT is None:
    print("over")
elif AT == ASK:
    print("the outcome says ask: the rest is bought from the continuer, in the cell below")
else:
    print(f"next: {AT}  ({BY_ID[AT].act})")

next: first-map  (hand_over)


**e.** And if the branch said `ask`, buy the rest. This is `panel/continuing.py`, the second prompt, with its own gate and its own checks and no repair.

In [24]:
from panel.continuing import continue_experience

if AT == ASK:
    began = time.time()
    CARRYING_ON, SPENT = await continue_experience(
        experience=DOCUMENT,
        after=M.id,
        came=CAME,
        reading={"came": CAME, "reading": str(DID.get("onIt", ""))},
        now=time.time(),
        pitch=ARGS["pitch"],
    )
    print(f"{time.time() - began:.1f} s · {len(CARRYING_ON.moments)} more moments\n")
    for one in CARRYING_ON.moments:
        print(f"{one.id:26} {one.act:10} {one.heading}")
    # Carry on walking into what was just bought.
    BY_ID |= {one.id: one for one in CARRYING_ON.moments}
    ORDER += [one.id for one in CARRYING_ON.moments]
    AT = CARRYING_ON.moments[0].id
else:
    print("no continuation was asked for")

no continuation was asked for


## 11. Changing a block

The blocks are the files in `agents/` and `shared/`. Edit one in the editor, run `bench.reload_prompts()`, and go back to section 4: the fingerprint changes only when the text being sent changed.

When one is settled: `python -m tools.prompts --write`, then `python -m pytest -q`, then `python -m research.run --iterations 4 --seed 0 --label <name>` — six households, four iterations, about an hour, and the eight axes are the answer to whether it improved.

⚠️ Do not run `pytest` and a run at the same time: `tests/test_trail.py` fails with `Event loop is closed` on contention, and it looks like a real regression.

In [25]:
print("fingerprint:", bench.reload_prompts())

fingerprint: bdbfb5b564f4
